# Prueba rápida de modelos Groq

Este notebook prueba los dos modelos usados por el onboarding:

- `openai/gpt-oss-20b`: agentes especializados.
- `openai/gpt-oss-120b`: agente orquestador.

> No escribas la API key directamente en el notebook. Guárdala en el archivo `.env` de la raíz del proyecto.

## 1. Instalar dependencias

Ejecuta esta celda solo si el entorno todavía no tiene instalados `groq` y `python-dotenv`.

In [ ]:
%pip install -q "groq>=0.31,<1" python-dotenv

## 2. Cargar configuración

El archivo `.env` debe contener:

```dotenv
GROQ_API_KEY=gsk_su_clave
GROQ_SPECIALIST_MODEL=openai/gpt-oss-20b
GROQ_ORCHESTRATOR_MODEL=openai/gpt-oss-120b
```

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from groq import Groq

# Funciona si Jupyter se abrió desde la raíz o desde la carpeta notebooks.
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(project_root / ".env")

api_key = os.getenv("GROQ_API_KEY")
if not api_key:
    raise RuntimeError("Falta GROQ_API_KEY en el archivo .env")

SPECIALIST_MODEL = os.getenv("GROQ_SPECIALIST_MODEL", "openai/gpt-oss-20b")
ORCHESTRATOR_MODEL = os.getenv("GROQ_ORCHESTRATOR_MODEL", "openai/gpt-oss-120b")
client = Groq(api_key=api_key, timeout=30.0, max_retries=1)

print("API key configurada: sí")
print("Modelo especialista:", SPECIALIST_MODEL)
print("Modelo orquestador:", ORCHESTRATOR_MODEL)

## 3. Función reutilizable con streaming

La función imprime los fragmentos a medida que Groq los entrega y también retorna el texto completo.

In [ ]:
def consultar_modelo(modelo: str, prompt: str) -> str:
    stream = client.chat.completions.create(
        model=modelo,
        messages=[
            {
                "role": "system",
                "content": "Responde en español, de forma clara y concisa. No muestres razonamiento interno.",
            },
            {"role": "user", "content": prompt},
        ],
        temperature=0.6,
        max_completion_tokens=512,
        top_p=0.95,
        reasoning_effort="medium",
        include_reasoning=False,
        stream=True,
        stop=None,
    )

    fragments = []
    for chunk in stream:
        content = chunk.choices[0].delta.content or ""
        print(content, end="", flush=True)
        fragments.append(content)
    print()
    return "".join(fragments).strip()

## 4. Probar el agente especialista — 20B

El ejemplo no contiene datos personales reales.

In [ ]:
respuesta_20b = consultar_modelo(
    SPECIALIST_MODEL,
    "Explica en máximo 35 palabras este resultado de identidad mock: verified=true, confidence=0.96",
)

## 5. Probar el agente orquestador — 120B

Este prompt reproduce una decisión simplificada de onboarding.

In [ ]:
respuesta_120b = consultar_modelo(
    ORCHESTRATOR_MODEL,
    """Evalúa el siguiente caso mock y responde únicamente con CONTINUE o ESCALATE, seguido de una explicación breve:
identity_verified=true
identity_confidence=0.96
risk_level=low
Reglas: identidad verificada, confianza mínima 0.8 y riesgo low para continuar.""",
)

## 6. Comparar resultados

Esta celda no realiza llamadas adicionales.

In [ ]:
print("--- Especialista 20B ---")
print(respuesta_20b)
print("\n--- Orquestador 120B ---")
print(respuesta_120b)

## Recomendaciones

- Usa solamente información mock; no envíes nombres, documentos o coincidencias AML reales.
- Si recibes `401`, revisa o rota `GROQ_API_KEY`.
- Si recibes `429`, espera el tiempo indicado por Groq antes de reintentar.
- Bajar `temperature` produce respuestas más consistentes para decisiones bancarias.